# ==========================================================
# Breast Cancer Survival Prediction using Apache Spark
# Notebook 03: Feature Engineering
# ==========================================================

"""
Objective
---------
1. Perform feature engineering on the clean SEER dataset.
2. Build new clinical features.
3. Prepare the target variable (Vital_Status binary).
4. Address remaining numerical missing values.
5. Export the final processed feature dataset.

Note
----
No model training.
No model evaluation.
"""

In [1]:
# 1. Import Libraries | Khai báo thư viện

print("=" * 60)
print("1. IMPORT LIBRARIES")
print("=" * 60)

import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Add project root directory to path | Thêm thư mục gốc dự án vào hệ thống
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import custom functions from src | Nạp các hàm tự định nghĩa từ src
from src.data.loader import create_spark_session, load_csv

1. IMPORT LIBRARIES


In [2]:
# 2. Create Spark Session | Khởi tạo phiên làm việc Spark

print("=" * 60)
print("2. CREATE SPARK SESSION")
print("=" * 60)

spark = create_spark_session("SEER Breast Cancer Feature Engineering")

2. CREATE SPARK SESSION


In [3]:
# 3. Load Clean Dataset | Tải tập dữ liệu đã làm sạch từ Notebook 02

print("=" * 60)
print("3. LOAD CLEANED DATASET")
print("=" * 60)

# Synchronized perfectly with Notebook 02 output | Đường dẫn đồng bộ chính xác với Notebook 02
df = load_csv(spark, "../data/processed/seer_breast_cancer_clean/seer_breast_cancer_clean.csv")

print(f"Initial Rows    : {df.count():,}")
print(f"Initial Columns : {len(df.columns)}")

3. LOAD CLEANED DATASET
Initial Rows    : 456,087
Initial Columns : 24


In [4]:
# 4. Handle Target Variable | Thiết lập biến mục tiêu nhị phân

print("=" * 60)
print("4. DEFINE TARGET VARIABLE")
print("=" * 60)

# Transform Vital_Status: Alive -> 0, Dead -> 1 (Positive Class) | Chuyển đổi nhãn sinh tử về dạng nhị phân
df = df.withColumn(
    "label",
    F.when(F.col("Vital_Status") == "Alive", 0)
    .when(F.col("Vital_Status") == "Dead", 1)
    .otherwise(None)
)

# Remove original target column to prevent redundancy | Loại bỏ cột nhãn gốc
df = df.drop("Vital_Status")

# Verify label distribution | Kiểm soát tỷ lệ phân phối lớp mục tiêu
print("Target Variable (label) Distribution:")
df.groupBy("label").count().show()

4. DEFINE TARGET VARIABLE
Target Variable (label) Distribution:
+-----+------+
|label| count|
+-----+------+
|    1|168781|
|    0|287306|
+-----+------+



In [5]:
# 5. Extract Age Group | Phân nhóm độ tuổi bệnh nhân

print("=" * 60)
print("5. AGE GROUP EXTRACTION")
print("=" * 60)

# Categorize patient age into 4 clinical groups | Chia khoảng tuổi bệnh nhân thành 4 lớp lâm sàng
df = df.withColumn(
    "Age_Group",
    F.when(F.col("Age") < 40, "Under_40")
    .when((F.col("Age") >= 40) & (F.col("Age") < 60), "40_to_59")
    .when((F.col("Age") >= 60) & (F.col("Age") < 80), "60_to_79")
    .otherwise("80_and_Above")
)

print("Age Group Distribution:")
df.groupBy("Age_Group").count().show()

5. AGE GROUP EXTRACTION
Age Group Distribution:
+------------+------+
|   Age_Group| count|
+------------+------+
|80_and_Above|456087|
+------------+------+



In [6]:
# 6. Extract Tumor Size Group | Phân nhóm kích thước khối u (T-stage)

print("=" * 60)
print("6. TUMOR SIZE GROUP EXTRACTION")
print("=" * 60)

# Categorize tumor size according to AJCC guidelines | Phân nhóm kích thước u theo chuẩn lâm sàng AJCC
df = df.withColumn(
    "Tumor_Size_Group",
    F.when(F.col("Tumor_Size") <= 20, "T1_Micro")
    .when((F.col("Tumor_Size") > 20) & (F.col("Tumor_Size") <= 50), "T2_Medium")
    .when(F.col("Tumor_Size") > 50, "T3_Large")
    .otherwise("Unknown")
)

print("Tumor Size Group Distribution:")
df.groupBy("Tumor_Size_Group").count().show()

6. TUMOR SIZE GROUP EXTRACTION
Tumor Size Group Distribution:
+----------------+------+
|Tumor_Size_Group| count|
+----------------+------+
|         Unknown| 37004|
|        T1_Micro|243452|
|       T2_Medium|140029|
|        T3_Large| 35602|
+----------------+------+



In [7]:
# 7. Extract Node Involvement | Tính toán tỷ lệ hạch lympho di căn

print("=" * 60)
print("7. NODE INVOLVEMENT RATIO")
print("=" * 60)

# Define Node Ratio: positive nodes / examined nodes | Công thức: Số hạch dương tính / Số hạch khảo sát
df = df.withColumn(
    "Node_Ratio",
    F.when(
        (F.col("Regional_Nodes_Examined") > 0) & (F.col("Regional_Nodes_Positive").isNotNull()),
        F.col("Regional_Nodes_Positive") / F.col("Regional_Nodes_Examined")
    ).otherwise(0.0)
)

# Cap node ratio logically at 1.0 | Khống chế cận trên của tỷ lệ hạch ở mức tối đa 1.0
df = df.withColumn(
    "Node_Ratio",
    F.when(F.col("Node_Ratio") > 1.0, 1.0).otherwise(F.col("Node_Ratio"))
)

df.select("Regional_Nodes_Positive", "Regional_Nodes_Examined", "Node_Ratio").show(10)

7. NODE INVOLVEMENT RATIO
+-----------------------+-----------------------+-------------------+
|Regional_Nodes_Positive|Regional_Nodes_Examined|         Node_Ratio|
+-----------------------+-----------------------+-------------------+
|                    1.0|                    7.0|0.14285714285714285|
|                    0.0|                    1.0|                0.0|
|                    0.0|                    1.0|                0.0|
|                   13.0|                   22.0| 0.5909090909090909|
|                    0.0|                    1.0|                0.0|
|                    9.0|                   10.0|                0.9|
|                   NULL|                    0.0|                0.0|
|                    0.0|                    3.0|                0.0|
|                   NULL|                    0.0|                0.0|
|                   NULL|                    0.0|                0.0|
+-----------------------+-----------------------+---------------

In [8]:
# 8. Extract Hormone Receptor Status | Thiết lập trạng thái thụ thể hormone nội tiết

print("=" * 60)
print("8. HORMONE RECEPTOR COMBINED STATUS")
print("=" * 60)

# Combine PR and ER Status into clinical categories | Tổ hợp hai chỉ số thụ thể ER và PR thành các nhánh lâm sàng
df = df.withColumn(
    "Hormone_Status",
    F.when((F.col("ER_Status") == "Positive") & (F.col("PR_Status") == "Positive"), "HR_Positive")
    .when((F.col("ER_Status") == "Negative") & (F.col("PR_Status") == "Negative"), "HR_Negative")
    .otherwise("HR_Mixed")
)

print("Hormone Combined Status Distribution:")
df.groupBy("Hormone_Status").count().show()

8. HORMONE RECEPTOR COMBINED STATUS
Hormone Combined Status Distribution:
+--------------+------+
|Hormone_Status| count|
+--------------+------+
|   HR_Negative| 77697|
|   HR_Positive|288396|
|      HR_Mixed| 89994|
+--------------+------+



In [9]:
# 9. Handle Numerical Missing Values | Xử lý giá trị khuyết thuộc tính số

print("=" * 60)
print("9. NULL IMPUTATION FOR NUMERICAL FEATURES")
print("=" * 60)

# Perform Median Imputation on key numerical attributes | Tính toán trung vị để điền rỗng thuộc tính số
numerical_features = ["Tumor_Size", "Regional_Nodes_Examined", "Regional_Nodes_Positive"]

for col in numerical_features:
    median_val = df.approxQuantile(col, [0.5], 0.01)[0]
    df = df.fillna({col: median_val})
    print(f"Column: {col:<25} | Imputed Nulls with Median: {median_val}")

9. NULL IMPUTATION FOR NUMERICAL FEATURES
Column: Tumor_Size                | Imputed Nulls with Median: 18.0
Column: Regional_Nodes_Examined   | Imputed Nulls with Median: 3.0
Column: Regional_Nodes_Positive   | Imputed Nulls with Median: 0.0


In [10]:
# 10. Type Casting & RegEx for Age | Ép kiểu an toàn & Làm sạch cột Age
# ==========================================================
print("=" * 60)
print("10. TYPE CASTING & ROBUST REGEX FOR AGE")
print("=" * 60)

# 1. Xử lý bóc tách số cho cột Age bằng RegEx trước khi Cast (Tránh phát sinh Null)
df = df.withColumn("Age_Cleaned", F.regexp_extract(F.col("Age"), r"(\d+)", 1))
df = df.withColumn("Age", F.col("Age_Cleaned").cast("double")).drop("Age_Cleaned")

# 2. Ép kiểu các thuộc tính danh mục phức tạp sang String
df = df.withColumn("Histologic_Type", F.col("Histologic_Type").cast("string"))
df = df.withColumn("Surgery_Primary_Site", F.col("Surgery_Primary_Site").cast("string"))

print("Data type casting and Age RegEx extraction finalized successfully.")

10. TYPE CASTING & ROBUST REGEX FOR AGE
Data type casting and Age RegEx extraction finalized successfully.


In [11]:
# 11. Removing Intermediary Columns | Loại bỏ các cột trung gian thừa

print("=" * 60)
print("11. REMOVING INTERMEDIARY COLUMNS")
print("=" * 60)

intermediate_cols = ["ER_Status", "PR_Status"]
df = df.drop(*intermediate_cols)

print(f"Dropped redundant intermediary columns: {intermediate_cols}")

11. REMOVING INTERMEDIARY COLUMNS
Dropped redundant intermediary columns: ['ER_Status', 'PR_Status']


In [12]:
# 12. Validate Processed Features | Xác thực cấu trúc dữ liệu thành phẩm

print("=" * 60)
print("12. FEATURE DATASET VALIDATION")
print("=" * 60)

print(f"Final Features Rows    : {df.count():,}")
print(f"Final Features Columns : {len(df.columns)}")
df.printSchema()

# Ensure no NULLs remain in critical numerical features
critical_cols = ["Age", "Tumor_Size", "Regional_Nodes_Examined", "Regional_Nodes_Positive", "Node_Ratio"]
for column in critical_cols:
    null_count = df.filter(F.col(column).isNull()).count()
    if null_count > 0:
        print(f"Alert: {column:30} has {null_count:,} nulls remaining!")
    else:
        print(f"Success: {column:30} is 100% clean.")

12. FEATURE DATASET VALIDATION
Final Features Rows    : 456,087
Final Features Columns : 26
root
 |-- Age: double (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race: string (nullable = true)
 |-- Marital_Status: string (nullable = true)
 |-- Tumor_Size: double (nullable = false)
 |-- Grade: string (nullable = true)
 |-- AJCC_T: string (nullable = true)
 |-- AJCC_N: string (nullable = true)
 |-- Regional_Nodes_Examined: double (nullable = false)
 |-- Regional_Nodes_Positive: double (nullable = false)
 |-- Sequence_Number: string (nullable = true)
 |-- Histologic_Type: string (nullable = true)
 |-- Laterality: string (nullable = true)
 |-- Diagnostic_Confirmation: string (nullable = true)
 |-- AJCC_M: string (nullable = true)
 |-- Surgery_Primary_Site: string (nullable = true)
 |-- Surgery_Other_Regional: string (nullable = true)
 |-- Surgery_Radiation_Sequence: string (nullable = true)
 |-- Radiation: string (nullable = true)
 |-- Chemotherapy: string (nullable = true)
 |-- 

In [13]:
# 13. Export Process-Ready Dataset | Ghi tập thuộc tính ra đĩa cứng

print("=" * 60)
print("13. EXPORT PROCESS-READY DATASET")
print("=" * 60)

output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "seer_breast_cancer_feature.csv")

print("Writing features dataset to CSV format...")
# Chuyển đổi sang Pandas và xuất file tĩnh (Đã bao gồm cột Age sạch)
df_pd = df.toPandas()
df_pd.to_csv(output_file, index=False, encoding="utf-8")

print(f"Feature engineering dataset exported successfully to: {output_file}")
print("\nProceed to Notebook 04 — Model Training")

13. EXPORT PROCESS-READY DATASET
Writing features dataset to CSV format...
Feature engineering dataset exported successfully to: ../data/processed\seer_breast_cancer_feature.csv

Proceed to Notebook 04 — Model Training


In [14]:
# Chạy thử đoạn này trong Notebook 03 để đối chiếu
df.select("Age").distinct().show(20, truncate=False)

+----+
|Age |
+----+
|70.0|
|75.0|
|35.0|
|80.0|
|25.0|
|85.0|
|50.0|
|45.0|
|60.0|
|10.0|
|40.0|
|30.0|
|20.0|
|15.0|
|55.0|
|65.0|
|90.0|
|1.0 |
|5.0 |
+----+

